In [1]:
import itertools

from tqdm.notebook import tqdm
import pandas as pd
import math
import numpy as np

total_datapoints = 2033
max_judge_queries = total_datapoints*20
failure_probability_delta = 0.1

def judge_query(p_r_pair):
    """
    Queries a prompt response pair to the judge LLM
    :param p_a_pair:
    :return:
    """

def data_reader(filename):
    """
    Reads the data from the json file into a pandas dataset
    :return:
    """
    return pd.read_json(filename, lines=True)

class UCB:
    def __init__(self, n_pairs, df):
        self.n_pairs = n_pairs
        self.responses = [np.empty(0) for _ in range(n_pairs)]          # Number of times each arm was pulled
        self.dataframe = df

    def select_pair(self):
        # If any pair hasn't been tried 4log(1/\delta), try it first
        for pair in range(self.n_pairs):
            if self.responses[pair].shape[0] < math.log(1/failure_probability_delta,2)*4:
                return pair

        ucb_values = [0.0] * self.n_pairs

        for pair in range(self.n_pairs):

            ucb_values[pair] = self.get_ucb_value(pair)

        return ucb_values.index(max(ucb_values))

    def get_ucb_value(self, pair_number):
        return self.get_sample_covariance(self.responses[pair_number])/(1-(2*math.sqrt(math.log(1/failure_probability_delta,2)/self.responses[pair_number].shape[0])))

    def get_sample_covariance(self, selected_pair_responses):
        return np.square(selected_pair_responses-selected_pair_responses.mean()).sum()/(selected_pair_responses.shape[0]-1)

    def update(self, chosen_pair):
        new_value = self.dataframe.iloc[chosen_pair*10+((self.responses[chosen_pair].shape[0])%10)]["predicted_score"]
        self.responses[chosen_pair] = np.append(self.responses[chosen_pair], new_value)

In [2]:
df = data_reader("helpsteer2.jsonl")
algorithm = UCB(total_datapoints, df)

In [11]:
df.iloc[8209]

prompt             Create 4 week program program overview and inc...
response           Here's a sample end-of-program project and rub...
human_score                                                        1
rationale          Content (40 points): The action plan provided ...
predicted_score                                                   26
Name: 8209, dtype: object

In [ ]:
for iteration in tqdm(range(int(max_judge_queries))):
    most_uncertain_pair = algorithm.select_pair()
    # if iteration < total_datapoints*2 and iteration % 10 == 0:
    #     print(most_uncertain_pair)
    if iteration > total_datapoints*14and iteration % 1000 == 0:
        print(most_uncertain_pair)
    algorithm.update(most_uncertain_pair)

C:\Users\aniket\AppData\Local\Temp\ipykernel_39768\121720781.py:5: SyntaxWarning: invalid decimal literal
  if iteration > total_datapoints*14and iteration % 1000 == 0:


KeyboardInterrupt: 

In [76]:
algorithm.get_ucb_value(2)

41.78831586787496

In [60]:
algorithm.responses[2]

array([2., 4.])

In [62]:
algorithm.get_sample_covariance(algorithm.responses[2])

2.0

In [65]:
1/failure_probability_delta

100.0

In [85]:
math.log(1/failure_probability_delta,2)*4

13.28771237954945